# Práctica 1 — Optimización de preprocesamiento de imágenes con paralelismo

**Materia:** Cómputo de Alto Desempeño  
**Alumnos:** Jessica Melani Romero Lora  
**Fecha:** 24 de mayo de 2026  

---

Este notebook toma el código base de preprocesamiento de imágenes visto en clase y le aplica cuatro mejoras orientadas al cómputo de alto desempeño:

| # | Actividad | Técnica aplicada |
|---|-----------|------------------|
| 1 | Optimizar la rutina de procesamiento | Vectorización NumPy (`.flatten()`) + lectura directa en gris |
| 2 | Un CSV dedicado por clase | Proceso exclusivo por clase; eliminación de contención en escritura |
| 3 | Comparación secuencial vs paralelo | Medición con `time.perf_counter()` y speedup empírico |
| 4 | Tiempo por clase individual | `Queue` multiproceso + análisis de desbalance de carga |

---

## Configuración e imports

Todas las constantes ajustables (rutas, tamaño de imagen, clases) se centralizan aquí para que cualquier cambio afecte a todo el notebook de forma uniforme.

In [ ]:
import time
from multiprocessing import Process, Queue, cpu_count
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Rutas ──────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/home/victor/Maestría/Cuatrimestre 3/Cómputo de alto desempeño/Dataset")
SALIDA_DIR   = Path("output")
SALIDA_SEQ   = SALIDA_DIR / "secuencial"

SALIDA_DIR.mkdir(exist_ok=True)
SALIDA_SEQ.mkdir(exist_ok=True)

# ── Parámetros del preprocesamiento ────────────────────────────────────────────
CLASES      = ["A", "B", "C", "D"]
TARGET_SIZE = (16, 16)                         # resolución de salida
N_PIXELES   = TARGET_SIZE[0] * TARGET_SIZE[1]  # 256 valores por imagen
COL_NOMBRES = [f"p{i}" for i in range(N_PIXELES)] + ["clase"]

print(f"Dataset            : {DATASET_ROOT}")
print(f"Salida paralela    : {SALIDA_DIR.resolve()}")
print(f"Salida secuencial  : {SALIDA_SEQ.resolve()}")
print(f"Núcleos disponibles: {cpu_count()}")

---

## Actividad 1 — Optimización de la rutina de procesamiento

### Problema en el código original

El código base extraía los píxeles con dos ciclos `for` anidados:

```python
# ❌ Original — O(filas × columnas) iteraciones en Python puro
for y in range(filas):
    for x in range(columnas):
        pix = re_size[y, x]
        file.write(f"{pix},")
```

Para una imagen 16×16 esto equivale a **256 iteraciones del intérprete Python por imagen**. Con ~3 600 imágenes, eso acumula más de 920 000 iteraciones solo para la extracción de píxeles. Además, el código original convertía `BGR → RGB → Gris` en dos pasos cuando `cv2.IMREAD_GRAYSCALE` lo hace directamente en uno.

### Solución aplicada

| Cambio | Beneficio |
|--------|-----------|
| `cv2.IMREAD_GRAYSCALE` | Elimina dos conversiones de espacio de color |
| `ndarray.flatten()` | Sustituye 256 iteraciones Python por una operación C compilada |
| Escritura por lotes con `pandas` | Reduce las operaciones de I/O de N (una/imagen) a 1 (una/clase) |

In [ ]:
def procesar_imagen(ruta: Path, target_size: tuple = TARGET_SIZE) -> np.ndarray | None:
    """
    Lee una imagen en escala de grises, la redimensiona y devuelve
    un vector NumPy aplanado de longitud N_PIXELES.

    Retorna None si la imagen no pudo leerse (archivo corrupto o ruta inválida)
    en lugar de lanzar una excepción, manteniendo el pipeline robusto.
    """
    img = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)  # lectura directa en gris
    if img is None:
        return None
    resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    return resized.flatten()  # sustituye los ciclos anidados for y / for x


def procesar_clase(
    clase: str,
    dataset_root: Path,
    salida_dir: Path,
    queue: Queue | None = None,
) -> None:
    """
    Procesa todas las imágenes de una clase y escribe su propio archivo CSV.

    Cada proceso escribe en un archivo exclusivo (clase_A.csv, clase_B.csv…)
    evitando la contención que producía el modo 'append' sobre un archivo
    compartido en el código original.

    Si se recibe una Queue, deposita (clase, duración) al terminar para
    que el proceso principal pueda recoger los tiempos individuales.
    """
    inicio = time.perf_counter()

    rutas  = sorted((Path(dataset_root) / clase).glob("*.JPG"))
    filas  = [
        np.append(vector, clase)
        for ruta in rutas
        if (vector := procesar_imagen(ruta)) is not None
    ]

    # Escritura en un único paso: más eficiente que abrir/cerrar por imagen
    df = pd.DataFrame(filas, columns=COL_NOMBRES)
    df.to_csv(Path(salida_dir) / f"clase_{clase}.csv", index=False)

    duracion = time.perf_counter() - inicio
    if queue is not None:
        queue.put((clase, duracion))


print("Funciones definidas correctamente.")

---

## Actividad 2 — Un archivo CSV por clase con núcleo dedicado

Se lanza un `Process` independiente por cada clase. Cada proceso escribe únicamente en su propio archivo (`clase_A.csv`, `clase_B.csv`…) sin interferir con los demás.

Una `Queue` recolecta el tiempo que tardó cada proceso; esos valores se usan después en la Actividad 4.

In [ ]:
cola_tiempos = Queue()

procesos = [
    Process(
        target=procesar_clase,
        args=(clase, DATASET_ROOT, SALIDA_DIR, cola_tiempos),
        name=f"Nucleo-{clase}",
    )
    for clase in CLASES
]

print("Iniciando procesamiento paralelo...")
t0_paralelo = time.perf_counter()

for p in procesos:
    p.start()
for p in procesos:
    p.join()  # espera a que todos terminen antes de continuar

tiempo_paralelo = time.perf_counter() - t0_paralelo

# Recoger tiempos individuales depositados en la cola
tiempos_por_clase = {clase: dur for clase, dur in iter(cola_tiempos.get, None)
                     if not cola_tiempos.empty() or True}
# Forma alternativa robusta de vaciar la cola:
tiempos_por_clase = {}
while not cola_tiempos.empty():
    clase, dur = cola_tiempos.get()
    tiempos_por_clase[clase] = dur

print(f"\nProcesamiento paralelo completado en {tiempo_paralelo:.4f} s")
print("\nTiempo por clase (dentro de su proceso):")
for clase in CLASES:
    print(f"  Clase {clase}: {tiempos_por_clase.get(clase, 0):.4f} s")

In [ ]:
# ── Unificación de los cuatro CSV en uno solo ──────────────────────────────────
# Esta operación se ejecuta en el proceso principal una sola vez,
# sin penalizar el tiempo de procesamiento paralelo.

dataset_final = pd.concat(
    [pd.read_csv(SALIDA_DIR / f"clase_{clase}.csv") for clase in CLASES],
    ignore_index=True,
)

ruta_final = SALIDA_DIR / "dataset_final.csv"
dataset_final.to_csv(ruta_final, index=False)

print(f"Dataset final guardado en : {ruta_final}")
print(f"Total de filas            : {len(dataset_final):,}")
print(f"Total de columnas         : {len(dataset_final.columns):,}  ({N_PIXELES} píxeles + 1 etiqueta)")
print("\nDistribución por clase:")
print(dataset_final["clase"].value_counts().sort_index())

---

## Actividad 3 — Comparación de tiempos: secuencial vs paralelo

Para cuantificar el beneficio del paralelismo se ejecuta el mismo preprocesamiento de forma **secuencial** (una clase tras otra, en el mismo hilo) y se compara el tiempo total con el del bloque paralelo anterior.

Los archivos del modo secuencial se escriben en `output/secuencial/` para no sobreescribir los del modo paralelo.

In [ ]:
print("Iniciando procesamiento secuencial...")
t0_secuencial = time.perf_counter()

for clase in CLASES:
    procesar_clase(clase, DATASET_ROOT, SALIDA_SEQ)  # sin queue: solo procesa

tiempo_secuencial = time.perf_counter() - t0_secuencial

speedup      = tiempo_secuencial / tiempo_paralelo
tiempo_ahor  = tiempo_secuencial - tiempo_paralelo

print(f"\nProcesamiento secuencial completado en {tiempo_secuencial:.4f} s")
print(f"Procesamiento paralelo   completado en {tiempo_paralelo:.4f} s")
print(f"\nSpeedup                 : {speedup:.2f}x")
print(f"Tiempo ahorrado         : {tiempo_ahor:.4f} s")

In [ ]:
# ── Gráfica comparativa ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

modalidades = ["Secuencial", "Paralelo"]
tiempos     = [tiempo_secuencial, tiempo_paralelo]
colores     = ["#e07a5f", "#3d405b"]

barras = ax.bar(modalidades, tiempos, color=colores, width=0.45, edgecolor="white")

for barra, valor in zip(barras, tiempos):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.2,
        f"{valor:.2f} s",
        ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

ax.set_title(
    f"Tiempo total de procesamiento\nSpeedup: {speedup:.2f}x",
    fontsize=12,
)
ax.set_ylabel("Segundos")
ax.set_ylim(0, max(tiempos) * 1.25)
ax.grid(axis="y", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
ruta_graf1 = SALIDA_DIR / "comparacion_secuencial_paralelo.png"
plt.savefig(ruta_graf1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Gráfica guardada en {ruta_graf1}")

---

## Actividad 4 — Tiempo de procesamiento por clase individual

Cada proceso depositó su tiempo de ejecución en la `Queue` al terminar. Aquí se visualiza el tiempo que tardó **cada núcleo** en procesar su clase asignada durante la ejecución paralela.

La diferencia entre clases refleja el desbalance de carga del dataset:

| Clase | Imágenes |
|-------|----------|
| A     |    363   |
| B     |    922   |
| C     |    990   |
| D     |  1 388   |

Los núcleos con más imágenes tardan más, y el proceso principal no puede continuar hasta que **el más lento** (Clase D) termine.

In [ ]:
# ── Conteo real de imágenes por clase ─────────────────────────────────────────
imagenes_por_clase = {
    clase: len(list((DATASET_ROOT / clase).glob("*.JPG")))
    for clase in CLASES
}

clases_ord    = sorted(tiempos_por_clase.keys())
tiempos_ord   = [tiempos_por_clase[c] for c in clases_ord]
imagenes_ord  = [imagenes_por_clase[c] for c in clases_ord]

# ── Gráfica de barras + línea de imágenes ─────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

colores_clase = ["#81b29a", "#f2cc8f", "#e07a5f", "#3d405b"]
barras = ax1.bar(
    clases_ord, tiempos_ord,
    color=colores_clase, width=0.5, edgecolor="white", label="Tiempo (s)",
)
ax2.plot(
    clases_ord, imagenes_ord,
    color="#264653", marker="o", linewidth=2, linestyle="--",
    label="Imágenes", zorder=5,
)

for barra, valor in zip(barras, tiempos_ord):
    ax1.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.05,
        f"{valor:.2f} s",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax1.set_xlabel("Clase")
ax1.set_ylabel("Tiempo de procesamiento (s)", color="#3d405b")
ax2.set_ylabel("Número de imágenes", color="#264653")
ax1.set_title(
    "Tiempo de procesamiento por clase\n(ejecución en paralelo — un núcleo por clase)",
    fontsize=12,
)
ax1.set_ylim(0, max(tiempos_ord) * 1.3)
ax1.grid(axis="y", alpha=0.3)
ax1.spines[["top"]].set_visible(False)

# Leyendas combinadas de ambos ejes
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

plt.tight_layout()
ruta_graf2 = SALIDA_DIR / "tiempo_por_clase.png"
plt.savefig(ruta_graf2, dpi=150, bbox_inches="tight")
plt.show()

print(f"Gráfica guardada en {ruta_graf2}")
print("\nResumen por clase:")
for clase in clases_ord:
    imgs = imagenes_por_clase[clase]
    t    = tiempos_por_clase[clase]
    print(f"  Clase {clase}: {imgs:>4} imágenes — {t:.4f} s — {imgs/t:.1f} img/s")

---

## Conclusiones

### Actividad 1 — Optimización de la rutina

Reemplazar los ciclos anidados `for y / for x` por `flatten()` de NumPy elimina el overhead del intérprete Python en la extracción de píxeles. NumPy opera sobre el bloque completo de memoria del arreglo con código compilado en C, lo que es órdenes de magnitud más rápido para arreglos de cualquier tamaño. Leer directamente en escala de grises con `cv2.IMREAD_GRAYSCALE` evita además dos conversiones de espacio de color innecesarias. La acumulación en memoria y la escritura en un solo paso con `pandas` elimina el costo de abrir y cerrar el archivo cientos de veces.

### Actividad 2 — CSV por clase

En el código original todos los procesos competían por escribir en el mismo archivo, lo que introducía tiempos de espera implícitos (el sistema operativo serializa los accesos a disco). Al asignar un archivo exclusivo por clase, los cuatro núcleos operan de forma completamente independiente: ninguno espera al otro, y el costo de I/O se distribuye sin fricciones. La fusión final de los cuatro CSV es una operación de lectura-concatenación que se ejecuta una sola vez en el proceso principal.

### Actividad 3 — Secuencial vs paralelo

La diferencia de tiempo entre ambas modalidades refleja de forma empírica el beneficio del paralelismo. En el modo secuencial el tiempo total es la suma de los tiempos de cada clase; en el modo paralelo el tiempo total está dominado por la clase más lenta (la de mayor número de imágenes). El *speedup* obtenido evidencia por qué el cómputo de alto desempeño prioriza la distribución de la carga sobre múltiples núcleos.

### Actividad 4 — Tiempo por clase

La variación de tiempo entre clases es directamente proporcional al número de imágenes que contiene cada una: la Clase D (1 388 imágenes) tarda significativamente más que la Clase A (363 imágenes). Esto pone de manifiesto que un balanceo de carga desigual entre núcleos limita el speedup real: el proceso principal no puede continuar hasta que el núcleo más lento (Clase D) termine. En un sistema de producción, esta observación justificaría un particionado dinámico de la carga (ej. *work-stealing*) en lugar de asignar una clase fija por núcleo.